In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
    precision_score,
)

In [2]:
# Carregar base e remover linhas com NaN
df_flights_limpas_0 = pd.read_csv("data/flights_limpas.csv")
df_flights_limpas = df_flights_limpas_0.dropna().copy()

# Features para o KMeans (sem alvo e sem ARRIVAL_DELAY para evitar leakage)
X = df_flights_limpas.drop(columns=['BOL_DELAYED', 'ARRIVAL_DELAY']).copy()
y = df_flights_limpas['BOL_DELAYED'].astype(int)

# Features ciclicas de horario ajudam separar padroes temporais no clustering
def hhmm_to_minutes(series):
    s = series.astype(int)
    hh = s // 100
    mm = s % 100
    return (hh * 60 + mm).astype('float32')

for col in ['SCHEDULED_DEPARTURE', 'SCHEDULED_ARRIVAL', 'WHEELS_OFF', 'WHEELS_ON']:
    minutes = hhmm_to_minutes(X[col])
    angle = 2 * np.pi * (minutes / 1440.0)
    X[f'{col}_SIN'] = np.sin(angle).astype('float32')
    X[f'{col}_COS'] = np.cos(angle).astype('float32')

X = X.astype('float32')

print('Shape X:', X.shape)
print('Distribuicao y:')
print(y.value_counts(normalize=True).sort_index())

Shape X: (4650569, 18)
Distribuicao y:
BOL_DELAYED
0    0.752789
1    0.247211
Name: proportion, dtype: float64


In [3]:
# KMeans final com melhores parametros (foco em acuracia geral)
BEST_K = 10
BEST_THRESHOLD = 0.3207

# Split treino/teste para avaliacao fora da amostra
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Padronizacao
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Amostra para acelerar ajuste do clustering
sample_size = min(300000, X_train_scaled.shape[0])
rng = np.random.default_rng(42)
sample_idx = rng.choice(X_train_scaled.shape[0], size=sample_size, replace=False)
X_train_sample = X_train_scaled[sample_idx]

# Treino do MiniBatchKMeans com parametros finais
kmeans_final = MiniBatchKMeans(
    n_clusters=BEST_K,
    random_state=42,
    batch_size=4096,
    n_init=10,
    max_iter=200,
)
kmeans_final.fit(X_train_sample)

# Clusters previstos
cluster_train = kmeans_final.predict(X_train_scaled)
cluster_test = kmeans_final.predict(X_test_scaled)

# Taxa de atraso por cluster no treino
train_df = pd.DataFrame({'cluster': cluster_train, 'y': y_train.values})
delayed_rate = train_df.groupby('cluster')['y'].mean().sort_values()

# Mapeamento cluster -> classe com limiar fixo
cluster_to_class = {c: int(rate >= BEST_THRESHOLD) for c, rate in delayed_rate.items()}
y_pred = pd.Series(cluster_test).map(cluster_to_class).fillna(0).astype(int)

# Metricas finais
acc = accuracy_score(y_test, y_pred)
prec1 = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
f1_1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
bacc = balanced_accuracy_score(y_test, y_pred)

print('Modelo final: MiniBatchKMeans')
print(f'Parametros: k={BEST_K}, limiar={BEST_THRESHOLD}')
print('\nTaxa de atraso por cluster (treino):')
print(delayed_rate)
print('\nMetricas no teste:')
print(f'Accuracy: {acc:.4f}')
print(f'Precision classe 1: {prec1:.4f}')
print(f'F1 classe 1: {f1_1:.4f}')
print(f'Balanced accuracy: {bacc:.4f}')
print('\nMatriz de confusao (teste):')
print(confusion_matrix(y_test, y_pred))
print('\nRelatorio de classificacao (teste):')
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Modelo final: MiniBatchKMeans
Parametros: k=10, limiar=0.3207

Taxa de atraso por cluster (treino):
cluster
4    0.179901
8    0.208810
0    0.223466
1    0.228376
3    0.244612
7    0.252894
5    0.253541
9    0.267174
6    0.270164
2    0.748977
Name: y, dtype: float64

Metricas no teste:
Accuracy: 0.7703
Precision classe 1: 0.7475
F1 classe 1: 0.1872
Balanced accuracy: 0.5476

Matriz de confusao (teste):
[[691869   8311]
 [205330  24604]]

Relatorio de classificacao (teste):
              precision    recall  f1-score   support

           0     0.7711    0.9881    0.8663    700180
           1     0.7475    0.1070    0.1872    229934

    accuracy                         0.7703    930114
   macro avg     0.7593    0.5476    0.5267    930114
weighted avg     0.7653    0.7703    0.6984    930114

